# NYC facade typologies: clustering buildings by facade materials and inspection history

**Team:** Amelia, Jerome, Kazuko, Sahil
**Course:** AI Methods I – Day 3: Unsupervised learning (IAAC MaAI 2026/27)

## 0. Introduction and problem definition

In New York City, every building taller than 6 stories must have its facade inspected every 5 years (FISP, Local Law 11). Each inspection ends with a status:
**SAFE**, **SWARMP** (safe with a repair and maintenance programme) or **UNSAFE**. On Day 2 we predicted the UNSAFE status. Today there is no target:
we ask **which kinds of buildings exist in the FISP stock**, judged by **what their facades are made of** and **how they have behaved over the inspection cycles**.

**Use case.** A facade team at the NYC Department of Buildings (DOB), or a facade consultant with a portfolio of buildings, wants to plan
**programmes per building type**: inspection guidance, repair specifications, outreach campaigns. A terra-cotta pre-war tower, a glass curtain wall
and a stucco-clad apartment block need different specialists, and a building that is UNSAFE in every cycle needs enforcement more than guidance.

- **Who:** DOB facade unit, facade engineering consultants, retrofit programme planners.
- **For what:** to split the FISP stock into a handful of groups, each with its own inspection and repair programme.
- **A useful group** is one we can **name** from its materials and history, and for which a **different action** makes sense.

**Unit of analysis: one building** (identified by its BIN, the DOB Building Identification Number). The cleaned Day 2 table has one row per
**filing** (one report per building and cycle, plus amended and follow-up reports). We aggregate the filings of each building into one row (section 1).

**Data.** We start from the cleaned table of our Day 2 assignment (NYC Open Data, *DOB NOW: Safety – Facades Compliance Filings*,
[data.cityofnewyork.us/d/xubg-57si](https://data.cityofnewyork.us/d/xubg-57si), downloaded 2026-10-23). The Day 2 cleaning notebook produced it; we
committed a reduced copy (30 columns, without addresses) to `data/facades_clean.csv.gz` (1.7 MB), so this notebook runs on its own.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("..") / "data"
FIG_DIR = Path("..") / "figures"
MODEL_DIR = Path("..") / "models"
DATASET_DIR = Path("..") / "dataset"
for d in (FIG_DIR, MODEL_DIR, DATASET_DIR):
    d.mkdir(exist_ok=True)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
filings = pd.read_csv(DATA_DIR / "facades_clean.csv.gz")
print(filings.shape)
filings.head()

In [ ]:
print(filings["FILING_TYPE"].value_counts(), "\n")
print(filings["FILING_STATUS"].value_counts(), "\n")
print("Buildings (BIN):", filings["BIN"].nunique(), "| rows without BIN:", filings["BIN"].isna().sum())

The table holds **67,659 filings**. The *INITIAL* filings are the regular 5-yearly reports; *AMENDED* and *SUBSEQUENT* filings come later, after
repairs or after an UNSAFE result. Each building appears once per cycle (cycles 6 to 10, about 2001–2026) with its initial filing, plus any
follow-ups. This is a time series of reports per building, so the natural unit for "what kind of building is this?" is **one building**, not one report.

## 1. Features and preprocessing

### 1.1 From filings to buildings

For each building (BIN) we compute:

| Feature | How | Meaning |
|---|---|---|
| `n_cycles` | distinct cycles with an initial filing | how long the building has been in the programme |
| `first_cycle`, `last_cycle`, `first_filing_year`, `last_filing_year` | min / max over the initial filings | when it entered and was last reported |
| `n_unsafe`, `n_swarmp`, `n_safe` | counts of initial filings per status | its inspection record |
| `share_unsafe`, `share_swarmp` | the counts divided by `n_cycles` | how often it was UNSAFE / needed repairs, independent of how long it was in the programme |
| `ever_unsafe` | 1 if `n_unsafe > 0` | at least one UNSAFE report |
| `latest_status` | status of the initial filing in its last cycle | where it stands today |
| `n_followups`, `followups_per_cycle` | count of AMENDED + SUBSEQUENT filings, and per cycle | how much repair paperwork the facade generates |
| `total_penalty`, `failure_to_correct_penalty`, `late_filing_penalty` | sums over all filings (USD) | how badly it complied |
| `log_penalty` | `log10(total_penalty + 1)` | the same on a readable scale (penalties range from 0 to 22 million USD) |
| `MATERIAL_*`, `WALL_TYPE_*` | 1 if **any** filing lists that material / wall type | what the facade is made of |
| `n_materials`, `n_wall_types` | number of materials / wall types listed | how mixed the facade is |
| `BOROUGH`, `BLOCK`, `LOT` | most frequent value | where it is (BBL tax lot) |

In [ ]:
filings = filings.dropna(subset=["BIN"]).copy()
filings["BIN"] = filings["BIN"].astype("int64")
MATERIALS = [c for c in filings if c.startswith("MATERIAL_")]
WALL_TYPES = [c for c in filings if c.startswith("WALL_TYPE_")]

initial = filings[filings["FILING_TYPE"] == "INITIAL"].sort_values(["BIN", "CYCLE"])
g_all = filings.groupby("BIN")
g_ini = initial.groupby("BIN")

buildings = pd.DataFrame({
    "BOROUGH": g_all["BOROUGH"].agg(lambda s: s.mode().iat[0]),
    "BLOCK": g_all["BLOCK"].agg(lambda s: s.mode().iat[0]),
    "LOT": g_all["LOT"].agg(lambda s: s.mode().iat[0]),
    "n_cycles": g_ini["CYCLE"].nunique(),
    "first_cycle": g_ini["CYCLE"].min(),
    "last_cycle": g_ini["CYCLE"].max(),
    "first_filing_year": g_ini["FILING_YEAR"].min(),
    "last_filing_year": g_ini["FILING_YEAR"].max(),
    "n_unsafe": g_ini["FILING_STATUS"].agg(lambda s: (s == "UNSAFE").sum()),
    "n_swarmp": g_ini["FILING_STATUS"].agg(lambda s: (s == "SWARMP").sum()),
    "n_safe": g_ini["FILING_STATUS"].agg(lambda s: (s == "SAFE").sum()),
    "latest_status": g_ini["FILING_STATUS"].last(),
    "n_followups": g_all["FILING_TYPE"].agg(lambda s: (s != "INITIAL").sum()),
    "total_penalty": g_all["TOTAL_PENALTY_AMT"].sum(),
    "failure_to_correct_penalty": g_all["FAILURE_TO_CORRECT_AMT"].sum(),
    "late_filing_penalty": g_all["LATE_FILING_AMT"].sum(),
})
buildings = buildings.join(g_all[MATERIALS + WALL_TYPES].max())
print("Buildings:", len(buildings), "| without an initial filing:", buildings["n_cycles"].isna().sum())

A few buildings have only amended / subsequent filings in the data (their initial report is outside the cleaned table). Without an initial filing we
cannot compute their inspection record, so we drop them. A few others list no material or wall type at all: we count those as 0 (nothing listed)
rather than drop them.

In [ ]:
buildings = buildings.dropna(subset=["n_cycles"]).copy()
missing_material = buildings[MATERIALS + WALL_TYPES].isna().any(axis=1).sum()
buildings[MATERIALS + WALL_TYPES] = buildings[MATERIALS + WALL_TYPES].fillna(0).astype(int)

int_cols = ["n_cycles", "first_cycle", "last_cycle", "first_filing_year", "last_filing_year",
            "n_unsafe", "n_swarmp", "n_safe", "n_followups", "BLOCK", "LOT"]
buildings[int_cols] = buildings[int_cols].astype(int)
buildings["share_unsafe"] = buildings["n_unsafe"] / buildings["n_cycles"]
buildings["share_swarmp"] = buildings["n_swarmp"] / buildings["n_cycles"]
buildings["ever_unsafe"] = (buildings["n_unsafe"] > 0).astype(int)
buildings["followups_per_cycle"] = buildings["n_followups"] / buildings["n_cycles"]
buildings["log_penalty"] = np.log10(buildings["total_penalty"] + 1)
buildings["n_materials"] = buildings[MATERIALS].sum(axis=1)
buildings["n_wall_types"] = buildings[WALL_TYPES].sum(axis=1)
buildings = buildings.reset_index()

print("Buildings kept:", len(buildings), "| filled 'no material listed':", missing_material)
print("Duplicated BIN:", buildings["BIN"].duplicated().sum(), "| missing values:", buildings.isna().sum().sum())
buildings.describe().T.round(2)

### 1.2 Which features define "similar"?

Choosing the columns is choosing the meaning of the clusters. For our use case (one programme per *type* of facade), two buildings are
similar when **their facades are made of the same things** and **they have behaved alike in inspections**. We cluster on 13 features:

- **Inspection record (4):** `share_unsafe`, `share_swarmp`, `followups_per_cycle`, `log_penalty`.
- **Time in the programme (1):** `n_cycles`.
- **Facade materials (8):** `n_materials` and the flags for masonry, terra cotta, natural stone, concrete, stucco / EIFS, glass and metal.

**Kept aside for interpretation** (not clustered on), to test whether the groups mean something:

- `BOROUGH`: location. If a group is concentrated in Manhattan without having seen the borough, that tells us something about the building stock.
- `WALL_TYPE_*`: the structural wall system (load-bearing masonry, cavity wall, steel frame, concrete frame...). It is a second description of
  construction, partly redundant with the materials, and it lets us check the material-based groups against something independent.
- `first_cycle` / `first_filing_year`: when the building entered the FISP programme (a rough proxy for newer or newly-tall buildings).
- `latest_status`, `ever_unsafe` and the penalty components: the current situation.

**Left out entirely:** `MATERIAL_OTHER` (a catch-all box on the form that says nothing about the facade) and the raw counts (`n_unsafe`, ...), which
mostly repeat `n_cycles` × share.

### 1.3 Scaling and transformations

- The features do **not** share a unit (shares in 0–1, counts, log-dollars, 0/1 flags), so we standardise them with a `StandardScaler` inside a
  `Pipeline`; otherwise `n_cycles` and `log_penalty` would dominate the distances.
- Penalties are extremely skewed (median 28,500 USD, maximum 22.6 million): we use `log10(total + 1)`.
- `followups_per_cycle` has a long tail too (up to 7), but very few buildings are above 2; we keep it as is, since the scaled values stay moderate.
- `BOROUGH` is categorical and is kept aside for interpretation rather than one-hot encoded.

In [ ]:
FEATURES = [
    "share_unsafe", "share_swarmp", "followups_per_cycle", "log_penalty", "n_cycles", "n_materials",
    "MATERIAL_MASONRY", "MATERIAL_TERRA_COTTA", "MATERIAL_NATURAL_STONE", "MATERIAL_CONCRETE",
    "MATERIAL_STUCCO_EIFS", "MATERIAL_GLASS", "MATERIAL_METAL",
]
X = buildings[FEATURES].astype(float)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), ["share_unsafe", "share_swarmp", "followups_per_cycle", "log_penalty", "n_cycles", "n_materials"]):
    ax.hist(X[col], bins=30, color="#4C72B0")
    ax.set_title(col)
fig.suptitle("Distributions of the numeric features clustered on (buildings, n = %d)" % len(X))
fig.tight_layout()
fig.savefig(FIG_DIR / "01_feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

X[FEATURES[6:]].mean().sort_values().plot.barh(figsize=(7, 3), color="#55A868", title="Share of buildings listing each material")
plt.xlabel("share of buildings")
plt.show()

Most buildings list **masonry** (91%); the other materials are each present in 11–25% of buildings. Half the buildings were **never UNSAFE**
(`share_unsafe` = 0), and most have been in 3–4 cycles. `log_penalty` is bimodal: about a third of buildings never paid a penalty (0), the others
paid typically 10,000–100,000 USD (4–5 on the log scale).

The scaler, and later the clustering model, are fitted inside a `Pipeline`, so the saved model takes the raw 13 columns.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled.shape

## 2. Dimensionality reduction

### 2.1 PCA

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE).fit(X_scaled)
evr = pca_full.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, len(evr) + 1), evr, color="#4C72B0", label="per component")
ax.plot(range(1, len(evr) + 1), evr.cumsum(), "o-", color="#C44E52", label="cumulative")
ax.axhline(0.8, ls="--", color="grey", lw=1)
ax.set_xlabel("principal component")
ax.set_ylabel("explained variance ratio")
ax.set_title("PCA: explained variance (13 standardised features)")
ax.set_xticks(range(1, len(evr) + 1))
ax.legend()
fig.savefig(FIG_DIR / "02_pca_explained_variance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Explained variance:", evr.round(3))
print("Cumulative:", evr.cumsum().round(3))
print("Components for 80%:", int(np.argmax(evr.cumsum() >= 0.8)) + 1)

In [ ]:
loadings = pd.DataFrame(pca_full.components_[:4].T, index=FEATURES, columns=["PC1", "PC2", "PC3", "PC4"])
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(loadings, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-0.7, vmax=0.7, ax=ax)
ax.set_title("PCA loadings of the first four components")
fig.savefig(FIG_DIR / "03_pca_loadings.png", dpi=150, bbox_inches="tight")
plt.show()

**Reading the loadings** (see the printed values above; signs are arbitrary):

PCA_READING_PLACEHOLDER

### 2.2 t-SNE

We use **t-SNE** (scikit-learn) to draw the 15,700 buildings in 2D. We chose t-SNE over UMAP to avoid an extra dependency (umap-learn and numba)
in the Python 3.14 environment. Perplexity 50 suits a dataset of this size. **Remember:** t-SNE keeps neighbourhoods, but the **sizes of the
islands and the distances between them mean nothing**.

Many buildings share exactly the same 0/1 material pattern, so t-SNE draws tight "islands", one per common material combination.

In [ ]:
pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
pca_xy = pca2.fit_transform(X_scaled)

tsne = TSNE(n_components=2, perplexity=50, init="pca", random_state=RANDOM_STATE)
tsne_xy = tsne.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
axes[0].scatter(pca_xy[:, 0], pca_xy[:, 1], s=3, alpha=0.4, color="grey")
axes[0].set(title="PCA (PC1 vs PC2)", xlabel="PC1", ylabel="PC2")
sc = axes[1].scatter(tsne_xy[:, 0], tsne_xy[:, 1], s=3, c=buildings["share_unsafe"], cmap="magma_r")
axes[1].set(title="t-SNE coloured by share of UNSAFE cycles", xticks=[], yticks=[])
plt.colorbar(sc, ax=axes[1])
for borough, rows in buildings.groupby("BOROUGH"):
    axes[2].scatter(tsne_xy[rows.index, 0], tsne_xy[rows.index, 1], s=3, alpha=0.6, label=borough)
axes[2].set(title="t-SNE coloured by borough (not clustered on)", xticks=[], yticks=[])
axes[2].legend(markerscale=4, fontsize=8)
fig.savefig(FIG_DIR / "04_tsne_overview.png", dpi=150, bbox_inches="tight")
plt.show()

TSNE_READING_PLACEHOLDER

## 3. Clustering

### 3.1 K-Means: choosing k

We fit K-Means for k = 2 … 10 and plot the inertia (elbow) and the silhouette score (computed on a fixed random sample of 5,000 buildings to keep it fast).

In [ ]:
ks = range(2, 11)
inertias, silhouettes = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_, sample_size=5000, random_state=RANDOM_STATE))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(ks), inertias, "o-")
axes[0].set(title="Elbow: K-Means inertia", xlabel="k", ylabel="inertia")
axes[1].plot(list(ks), silhouettes, "o-", color="#C44E52")
axes[1].set(title="Silhouette score (5,000-building sample)", xlabel="k", ylabel="silhouette")
for ax in axes:
    ax.axvline(6, ls="--", color="grey", lw=1)
fig.savefig(FIG_DIR / "05_choice_of_k.png", dpi=150, bbox_inches="tight")
plt.show()
pd.DataFrame({"k": list(ks), "inertia": np.round(inertias), "silhouette": np.round(silhouettes, 3)})

CHOICE_K_PLACEHOLDER

In [ ]:
K = 6
kmeans = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(X_scaled)
labels_km = kmeans.labels_
print("Cluster sizes:", np.bincount(labels_km))
print("Silhouette (all buildings):", round(silhouette_score(X_scaled, labels_km), 3))

### 3.2 Hierarchical clustering (Ward)

Ward linkage on 15,700 buildings needs a distance matrix of about 1 GB, so we build the **dendrogram on a random sample of 4,000 buildings**, cut it
into the same number of clusters, and extend the labels to every building with a 5-nearest-neighbour classifier fitted on the sample. That gives a
Ward label for each building, which we can compare with K-Means.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
sample_idx = rng.choice(len(X_scaled), size=4000, replace=False)
X_sample = X_scaled[sample_idx]

Z = linkage(X_sample, method="ward")
fig, ax = plt.subplots(figsize=(12, 4.5))
dendrogram(Z, truncate_mode="lastp", p=30, ax=ax, color_threshold=Z[-(K - 1), 2])
ax.axhline(Z[-(K - 1), 2], ls="--", color="grey")
ax.set(title="Ward dendrogram (4,000-building sample, last 30 merges); dashed line = cut into %d clusters" % K,
       xlabel="cluster size (or building index)", ylabel="Ward distance")
fig.savefig(FIG_DIR / "06_ward_dendrogram.png", dpi=150, bbox_inches="tight")
plt.show()
print("Heights of the last 8 merges:", Z[-8:, 2].round(1))

In [ ]:
ward = AgglomerativeClustering(n_clusters=K, linkage="ward").fit(X_sample)
knn_ward = KNeighborsClassifier(n_neighbors=5).fit(X_sample, ward.labels_)
labels_ward = knn_ward.predict(X_scaled)

print("Ward cluster sizes:", np.bincount(labels_ward))
print("Silhouette Ward (all buildings):", round(silhouette_score(X_scaled, labels_ward), 3))
print("ARI K-Means vs Ward (all buildings):", round(adjusted_rand_score(labels_km, labels_ward), 3))
print("ARI K-Means vs Ward (the 4,000 sampled buildings):", round(adjusted_rand_score(labels_km[sample_idx], ward.labels_), 3))
pd.crosstab(pd.Series(labels_km, name="K-Means"), pd.Series(labels_ward, name="Ward"))

### 3.3 DBSCAN: is there dense structure and noise?

As a third view we try DBSCAN. We read `eps` from the k-distance plot (distance to the 10th neighbour, `min_samples = 10`).

In [ ]:
nn = NearestNeighbors(n_neighbors=10).fit(X_scaled)
kdist = np.sort(nn.kneighbors(X_scaled)[0][:, -1])
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(kdist)
ax.set(title="k-distance plot (10th nearest neighbour)", xlabel="buildings sorted by distance", ylabel="distance to 10th neighbour")
ax.axhline(1.5, ls="--", color="grey")
fig.savefig(FIG_DIR / "07_dbscan_kdistance.png", dpi=150, bbox_inches="tight")
plt.show()

for eps in (1.0, 1.5, 2.0):
    db = DBSCAN(eps=eps, min_samples=10).fit(X_scaled)
    n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    print(f"eps={eps}: {n_clusters} clusters, noise = {(db.labels_ == -1).mean():.1%}, "
          f"largest cluster = {np.bincount(db.labels_[db.labels_ >= 0]).max() / len(db.labels_):.1%}")

DBSCAN_READING_PLACEHOLDER

### 3.4 Comparison and final choice

COMPARISON_PLACEHOLDER

## 4. Stability

We check that the K-Means groups are not an accident of the random start or of the particular buildings in the data:

1. **Different seeds:** refit with 10 different `random_state` values and compare each result with our model (ARI).
2. **Subsamples:** refit on 10 random 80% subsamples and assign *all* buildings with the subsample model, then compare with our model (ARI).

In [ ]:
seed_ari = [adjusted_rand_score(labels_km, KMeans(n_clusters=K, n_init=10, random_state=s).fit_predict(X_scaled))
            for s in range(10)]
sub_ari = []
for s in range(10):
    idx = np.random.default_rng(s).choice(len(X_scaled), size=int(0.8 * len(X_scaled)), replace=False)
    km_sub = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(X_scaled[idx])
    sub_ari.append(adjusted_rand_score(labels_km, km_sub.predict(X_scaled)))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.boxplot([seed_ari, sub_ari], tick_labels=["10 random seeds", "10 subsamples (80%)"])
ax.set(ylabel="ARI with the final model", title="Stability of K-Means (k = %d)" % K, ylim=(0, 1.05))
fig.savefig(FIG_DIR / "08_stability.png", dpi=150, bbox_inches="tight")
plt.show()
print("Seeds: ARI mean %.3f, min %.3f" % (np.mean(seed_ari), np.min(seed_ari)))
print("Subsamples: ARI mean %.3f, min %.3f" % (np.mean(sub_ari), np.min(sub_ari)))

STABILITY_PLACEHOLDER

## 5. Interpretation

### 5.1 Profiles of the clusters

In [ ]:
buildings["cluster"] = labels_km
profile = buildings.groupby("cluster")[FEATURES].mean()
profile.insert(0, "share_of_buildings", buildings["cluster"].value_counts(normalize=True).sort_index())
profile.insert(0, "size", buildings["cluster"].value_counts().sort_index())
profile.round(2).T

In [ ]:
# z-scores of each cluster mean, relative to all buildings: what makes each group different
z = (buildings.groupby("cluster")[FEATURES].mean() - X.mean()) / X.std()
fig, ax = plt.subplots(figsize=(11, 4.5))
sns.heatmap(z, annot=buildings.groupby("cluster")[FEATURES].mean().round(2), fmt="", cmap="RdBu_r", center=0, vmin=-2, vmax=2,
            cbar_kws={"label": "difference from the average building (std)"}, ax=ax)
ax.set(title="Cluster profiles (numbers = cluster means; colour = standardised difference)", xlabel="", ylabel="cluster")
plt.xticks(rotation=40, ha="right")
fig.savefig(FIG_DIR / "09_cluster_profiles.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.2 Cross-tabulations with variables we did not cluster on

In [ ]:
aside = buildings.groupby("cluster").agg(
    median_penalty_usd=("total_penalty", "median"),
    mean_first_filing_year=("first_filing_year", "mean"),
    entered_after_cycle_6=("first_cycle", lambda s: (s > 6).mean()),
    ever_unsafe=("ever_unsafe", "mean"),
    latest_unsafe=("latest_status", lambda s: (s == "UNSAFE").mean()),
    latest_safe=("latest_status", lambda s: (s == "SAFE").mean()),
)
aside.round(2)

In [ ]:
borough_tab = pd.crosstab(buildings["cluster"], buildings["BOROUGH"], normalize="index")
wall_tab = buildings.groupby("cluster")[WALL_TYPES].mean()
wall_tab.columns = [c.replace("WALL_TYPE_", "").replace("_", " ").lower() for c in wall_tab.columns]

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), gridspec_kw={"width_ratios": [5, 8]})
sns.heatmap(borough_tab, annot=True, fmt=".0%", cmap="Blues", cbar=False, ax=axes[0])
axes[0].set(title="Borough (share of each cluster)", xlabel="", ylabel="cluster")
sns.heatmap(wall_tab, annot=True, fmt=".0%", cmap="Greens", cbar=False, ax=axes[1])
axes[1].set(title="Wall types listed (share of buildings in each cluster)", xlabel="", ylabel="")
for ax in axes:
    ax.tick_params(axis="x", rotation=35)
fig.savefig(FIG_DIR / "10_crosstab_borough_walltype.png", dpi=150, bbox_inches="tight")
plt.show()
print("All buildings by borough:\n", buildings["BOROUGH"].value_counts(normalize=True).round(2))

### 5.3 Naming the clusters

NAMING_PLACEHOLDER

In [ ]:
CLUSTER_NAMES = {c: f"cluster {c}" for c in range(K)}  # NAMES_TMP
buildings["cluster_name"] = buildings["cluster"].map(CLUSTER_NAMES)

summary = buildings.groupby(["cluster", "cluster_name"]).agg(
    buildings=("BIN", "size"),
    share_unsafe=("share_unsafe", "mean"),
    manhattan=("BOROUGH", lambda s: (s == "MANHATTAN").mean()),
).reset_index()
summary["share_of_buildings"] = summary["buildings"] / len(buildings)
summary.round(2)

In [ ]:
names_sorted = [CLUSTER_NAMES[c] for c in range(K)]
palette = dict(zip(names_sorted, sns.color_palette("tab10", K)))
fig, axes = plt.subplots(1, 2, figsize=(17, 6.5))
for name in names_sorted:
    m = (buildings["cluster_name"] == name).to_numpy()
    axes[0].scatter(pca_xy[m, 0], pca_xy[m, 1], s=3, alpha=0.5, color=palette[name], label=name)
    axes[1].scatter(tsne_xy[m, 0], tsne_xy[m, 1], s=3, alpha=0.5, color=palette[name], label=name)
axes[0].set(title="Clusters on the PCA map", xlabel="PC1", ylabel="PC2")
axes[1].set(title="Clusters on the t-SNE map (perplexity 50)", xticks=[], yticks=[])
axes[1].legend(markerscale=5, fontsize=9, loc="center left", bbox_to_anchor=(1, 0.5))
fig.tight_layout()
fig.savefig(FIG_DIR / "11_clusters_pca_tsne.png", dpi=150, bbox_inches="tight")
plt.show()

CLUSTERS_MAP_READING_PLACEHOLDER

## 6. Save and reuse the model

The final model is K-Means, which has `.predict()`, so the saved pipeline (scaler + K-Means) can assign any new building directly. We refit the
same pipeline on the raw 13 columns so that a user only needs to pass a `DataFrame` with those columns.

In [ ]:
model = make_pipeline(StandardScaler(), KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE))
model.fit(X)
assert adjusted_rand_score(labels_km, model.predict(X)) == 1.0  # same clusters as above
joblib.dump(model, MODEL_DIR / "model.joblib")
print("Saved", MODEL_DIR / "model.joblib", f"({(MODEL_DIR / 'model.joblib').stat().st_size / 1024:.0f} KB)")

**Reload and assign a hand-made example:** a pre-war Manhattan apartment building with a brick facade and terra-cotta and stone ornament,
in 4 cycles, SWARMP twice and SAFE twice, one amended filing, 15,000 USD of penalties.

In [ ]:
reloaded = joblib.load(MODEL_DIR / "model.joblib")
example = pd.DataFrame([{
    "share_unsafe": 0.0, "share_swarmp": 0.5, "followups_per_cycle": 0.25, "log_penalty": np.log10(15000 + 1),
    "n_cycles": 4, "n_materials": 3,
    "MATERIAL_MASONRY": 1, "MATERIAL_TERRA_COTTA": 1, "MATERIAL_NATURAL_STONE": 1, "MATERIAL_CONCRETE": 0,
    "MATERIAL_STUCCO_EIFS": 0, "MATERIAL_GLASS": 0, "MATERIAL_METAL": 0,
}])
c = int(reloaded.predict(example[FEATURES])[0])
print("Cluster", c, "->", CLUSTER_NAMES[c])

## 7. Export the clustered dataset

The published table has one row per building with: identifiers and location (`BIN`, `BOROUGH`, `BLOCK`, `LOT`; **no street addresses**), the
engineered building-level features, the material and wall-type flags, `cluster` and `cluster_name` (K-Means), `cluster_ward` (the Ward
labels, for comparison), and the map coordinates `pca_x`, `pca_y`, `tsne_x`, `tsne_y`.

In [ ]:
final = buildings.copy()
final["cluster_ward"] = labels_ward
final[["pca_x", "pca_y"]] = pca_xy
final[["tsne_x", "tsne_y"]] = tsne_xy

ordered = (["BIN", "BOROUGH", "BLOCK", "LOT",
            "n_cycles", "first_cycle", "last_cycle", "first_filing_year", "last_filing_year",
            "n_safe", "n_swarmp", "n_unsafe", "share_swarmp", "share_unsafe", "ever_unsafe", "latest_status",
            "n_followups", "followups_per_cycle",
            "total_penalty", "failure_to_correct_penalty", "late_filing_penalty", "log_penalty",
            "n_materials", "n_wall_types"] + MATERIALS + WALL_TYPES +
           ["cluster", "cluster_name", "cluster_ward", "pca_x", "pca_y", "tsne_x", "tsne_y"])
final = final[ordered]
for col in ["share_swarmp", "share_unsafe", "followups_per_cycle", "log_penalty", "pca_x", "pca_y", "tsne_x", "tsne_y"]:
    final[col] = final[col].round(4)

assert final["BIN"].is_unique and final.isna().sum().sum() == 0
final.to_csv(DATASET_DIR / "nyc_facade_typologies.csv", index=False)
print(final.shape)
final.head()

In [ ]:
# the figure shown in the dataset card
import shutil
shutil.copy(FIG_DIR / "11_clusters_pca_tsne.png", DATASET_DIR / "clusters_pca_tsne.png")
shutil.copy(FIG_DIR / "09_cluster_profiles.png", DATASET_DIR / "cluster_profiles.png")
shutil.copy(FIG_DIR / "09_cluster_profiles.png", MODEL_DIR / "cluster_profiles.png")

## 8. Conclusions

CONCLUSIONS_PLACEHOLDER